# 007 Handle Errors and Fallbacks in a Skill

这是第七课：给天气 Skill 加入错误处理和 fallback 教学。

学习目标：

1. 理解一份 Skill 什么时候要开始认真处理异常分支
2. 学会把“主路径”和“fallback 路径”讲清楚
3. 学会区分：澄清、失败、降级、fallback
4. 用真实天气 Skill 做一次错误处理和 fallback 演练

这节课继续使用：

- `.agents/skills/weather-query-assistant/`


## 先明确这节课解决什么问题

到第六课为止，这份天气 Skill 已经能跑一条完整小链路：

- 用户问题
- 地点判断
- 标准化地点
- 构建 `wttr.in` 查询 URL
- 生成可执行命令

但真实使用里，总会遇到这些问题：

- 地点缺失
- 地点很脏
- 用户明确要求 JSON
- 主路径不适合当前场景

这时你不能只会走 happy path，必须开始处理错误和 fallback。


## 先看当前真实 Skill

第七课不先加新目录，而是先看现有 Skill 是否已经具备 fallback 的概念。


In [1]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


In [2]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Normalize the location when the user input is noisy or inconsistently formatted.
4. Build a stable weather query string before calling the external service.
5. Use `wttr.in` as the primary source.
6. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
7. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
8. Do not guess when weather data is unavailable.

## References

- Read `referenc

## 从 `SKILL.md` 里读出三类路径

现在这份天气 Skill，至少已经隐含了 3 条路径：

1. 澄清路径：地点不明确 -> 先追问
2. 主路径：地点明确 -> `wttr.in`
3. fallback 路径：需要 JSON 或主路径不合适 -> Open-Meteo

这节课就是把这三条路径明确写出来。


In [3]:
path_types = {
    'clarify': 'location is missing or ambiguous',
    'primary': 'wttr.in for concise human-readable weather',
    'fallback': 'Open-Meteo for structured JSON-friendly output',
}

from pprint import pprint
pprint(path_types)


{'clarify': 'location is missing or ambiguous',
 'fallback': 'Open-Meteo for structured JSON-friendly output',
 'primary': 'wttr.in for concise human-readable weather'}


## 先定义一个教学版的决策函数

这里不追求做复杂业务引擎，只做一份清晰的 session router。

它负责判断：

- 先澄清？
- 走主路径？
- 走 fallback？


In [4]:
def decide_weather_path(
    user_question: str,
    location: str | None = None,
    wants_json: bool = False,
    primary_available: bool = True,
) -> dict:
    if not location:
        return {
            'path': 'clarify',
            'reason': 'location_missing',
            'assistant_reply': '请告诉我你要查询哪个城市或地点的天气。',
        }

    if wants_json:
        return {
            'path': 'fallback',
            'reason': 'json_requested',
            'source': 'open-meteo',
        }

    if not primary_available:
        return {
            'path': 'fallback',
            'reason': 'primary_unavailable',
            'source': 'open-meteo',
        }

    return {
        'path': 'primary',
        'reason': 'normal_weather_query',
        'source': 'wttr.in',
    }


## 先看最简单的澄清路径

这是最常见也最容易忽略的异常分支。


In [5]:
pprint(decide_weather_path('今天会下雨吗？'))


{'assistant_reply': '请告诉我你要查询哪个城市或地点的天气。',
 'path': 'clarify',
 'reason': 'location_missing'}


## 再看“用户明确要 JSON”的 fallback 路径

这里不是错误，而是“主路径不适合当前输出目标”。

这类 fallback 很常见。


In [6]:
pprint(decide_weather_path('给我北京当前天气的 JSON 结果', location='beijing', wants_json=True))


{'path': 'fallback', 'reason': 'json_requested', 'source': 'open-meteo'}


## 再看“主路径不可用”的 fallback 路径

这类 fallback 更像真实运行时降级。

比如：

- `wttr.in` 当前不适合用
- 或你想临时切走文本输出路径


In [7]:
pprint(decide_weather_path('帮我看下北京天气', location='beijing', primary_available=False))


{'path': 'fallback', 'reason': 'primary_unavailable', 'source': 'open-meteo'}


## 主路径依然是默认路径

如果没有特殊需求，也没有异常，就继续走 `wttr.in`。


In [8]:
pprint(decide_weather_path('帮我看下北京天气', location='beijing'))


{'path': 'primary', 'reason': 'normal_weather_query', 'source': 'wttr.in'}


## 把主路径和 fallback 路径接进已有脚本

这里我们继续复用前面几课的脚本。

- 主路径：继续构建 `wttr.in` 查询 URL
- fallback 路径：先返回结构化决策信息，不在这节课硬做完整 Open-Meteo 坐标链

这样边界更稳。


In [9]:
import subprocess


def build_primary_command(location: str, mode: str = 'compact') -> str:
    result = subprocess.run(
        [
            'python',
            '.agents/skills/weather-query-assistant/scripts/build_wttr_query.py',
            location,
            '--mode',
            mode,
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or 'build_wttr_query failed')
    return f'curl -s "{result.stdout.strip()}"'


def run_weather_session_with_fallback(
    user_question: str,
    location: str | None = None,
    wants_json: bool = False,
    primary_available: bool = True,
    mode: str = 'compact',
) -> dict:
    decision = decide_weather_path(
        user_question=user_question,
        location=location,
        wants_json=wants_json,
        primary_available=primary_available,
    )

    if decision['path'] == 'clarify':
        return {
            'decision': decision,
            'next_action': 'ask_clarification',
        }

    if decision['path'] == 'fallback':
        return {
            'decision': decision,
            'next_action': 'prepare_open_meteo_flow',
            'note': '下一步应该进入坐标查询和 Open-Meteo 请求链路。',
        }

    command = build_primary_command(location, mode=mode)
    return {
        'decision': decision,
        'next_action': 'run_primary_query',
        'command': command,
    }


## 跑一遍三种真实分支

这一格是第七课最重要的观察点。


In [10]:
examples = [
    {
        'title': 'clarify path',
        'kwargs': {'user_question': '今天会下雨吗？'},
    },
    {
        'title': 'primary path',
        'kwargs': {'user_question': '帮我看下北京天气', 'location': 'beijing', 'mode': 'current'},
    },
    {
        'title': 'fallback path',
        'kwargs': {'user_question': '给我北京天气的 JSON 结果', 'location': 'beijing', 'wants_json': True},
    },
]

for item in examples:
    print('case =', item['title'])
    pprint(run_weather_session_with_fallback(**item['kwargs']))
    print('-' * 60)


case = clarify path
{'decision': {'assistant_reply': '请告诉我你要查询哪个城市或地点的天气。',
              'path': 'clarify',
              'reason': 'location_missing'},
 'next_action': 'ask_clarification'}
------------------------------------------------------------
case = primary path
{'command': 'curl -s "https://wttr.in/Beijing?m&format=3"',
 'decision': {'path': 'primary',
              'reason': 'normal_weather_query',
              'source': 'wttr.in'},
 'next_action': 'run_primary_query'}
------------------------------------------------------------
case = fallback path
{'decision': {'path': 'fallback',
              'reason': 'json_requested',
              'source': 'open-meteo'},
 'next_action': 'prepare_open_meteo_flow',
 'note': '下一步应该进入坐标查询和 Open-Meteo 请求链路。'}
------------------------------------------------------------


## 区分四个概念

这一课里有 4 个容易混的词：

1. 澄清：信息不够，先追问
2. 失败：某一步真的执行不了
3. 降级：主路径不可用，切到次优路径
4. fallback：为另一类输出目标准备的替代路径

你现在至少要能把这四个概念分开。


In [11]:
concepts = {
    'clarify': 'input is incomplete',
    'failure': 'an action cannot complete successfully',
    'degrade': 'switch away from the preferred path because it is unavailable',
    'fallback': 'use an alternative path that better fits the current need',
}

pprint(concepts)


{'clarify': 'input is incomplete',
 'degrade': 'switch away from the preferred path because it is unavailable',
 'failure': 'an action cannot complete successfully',
 'fallback': 'use an alternative path that better fits the current need'}


## 为什么这节课不急着做完整 Open-Meteo 链路

因为这节课的重点是“分支决策”，不是再扩一个完整外部集成。

如果这一课就把：

- 地理编码
- 坐标解析
- Open-Meteo 请求
- 返回字段映射

全加进来，主线会被冲散。

所以这节课只把 fallback 决策边界讲清楚。


In [12]:
why_stop_here = [
    'focus this lesson on branching logic',
    'avoid mixing routing with full external integration',
    'keep fallback concept clear before implementation grows',
]

pprint(why_stop_here)


['focus this lesson on branching logic',
 'avoid mixing routing with full external integration',
 'keep fallback concept clear before implementation grows']


## 这节课的正式开发价值

第七课的核心不是多写几行代码，而是开始让 Skill 具备“异常分支意识”。

一份真实可用的 Skill，不能只会成功路径。

它必须知道：

- 什么时候先问用户
- 什么时候继续主路径
- 什么时候切 fallback


## 如果继续往下走，还剩多少期比较合适

如果按当前这条天气 Skill 教学线，我建议总共做 10 课比较合适。

现在已经完成：

1. 读真实 Skill
2. 新建最小 Skill
3. 引入 references
4. 引入 scripts
5. 接成 mini workflow
6. 跑完整小会话
7. 处理错误和 fallback

所以还建议保留 3 课：

8. 做一次真正的 Open-Meteo fallback 实现
9. 给这份 Skill 增加测试 / 验证思路
10. 做一次完整收尾：复盘这份 Skill 是怎么从 0 长到可用的


In [13]:
remaining_lessons = {
    'total_recommended': 10,
    'completed_now': 7,
    'remaining': 3,
    'next_three': [
        '008 implement a real Open-Meteo fallback path',
        '009 add validation and testing ideas for the skill',
        '010 final review and evolution recap',
    ],
}

pprint(remaining_lessons)


{'completed_now': 7,
 'next_three': ['008 implement a real Open-Meteo fallback path',
                '009 add validation and testing ideas for the skill',
                '010 final review and evolution recap'],
 'remaining': 3,
 'total_recommended': 10}


## 当前阶段结论

你现在需要记住：

1. 第七课的核心是让 Skill 开始具备错误处理和 fallback 意识
2. 澄清、失败、降级、fallback 是四个不同概念
3. 这份天气 Skill 现在已经不只是能跑 happy path，而是开始能管理多条分支
4. 这一课先把决策边界讲清楚，比一口气把 Open-Meteo 全接完更重要
5. 如果按当前路线继续，整个天气 Skill 教学线建议总共 10 课，现在还剩 3 课

下一步建议：

- 继续第八课：实现真正的 Open-Meteo fallback 路径
